# Attention Is All You Need
## Experiment Notebook

**Paper**: Vaswani et al. (NeurIPS 2017) | [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)

This notebook demonstrates:
1. Transformer architecture construction and verification
2. WMT14 EN-DE data pipeline with BPE tokenization
3. Training with the Noam LR schedule and label smoothing
4. Greedy and beam search inference
5. Attention weight visualization
6. BLEU score evaluation

Uses `quick_test()` configuration for fast demonstration.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import numpy as np
import matplotlib.pyplot as plt

from transformer_mt import (
    ExperimentConfig,
    Transformer,
    create_masks,
    get_dataloaders,
    train_tokenizer,
    beam_search,
    greedy_decode,
    train,
    set_seed,
)
from transformer_mt.models.attention import ScaledDotProductAttention
from transformer_mt.training.trainer import compute_bleu
from transformer_mt.data.tokenizer import PAD_IDX, BOS_IDX, EOS_IDX

# load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

plt.rcParams["figure.figsize"] = (12, 6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration

Using `quick_test()` for fast demonstration: d_model=64, n_layers=2, n_heads=4, vocab_size=1000, total_steps=500.

In [ ]:
config = ExperimentConfig.quick_test()
set_seed(config.train.seed)

print("Model Config:", config.model)
print("Data Config: ", config.data)
print("Train Config:", config.train)

## 3. Data: BPE Tokenizer and WMT14 EN-DE

Train a shared BPE tokenizer on WMT14 data, then create DataLoaders with token-count batching (Section 5.1).

> "Sentences were encoded using byte-pair encoding, which has a shared source-target vocabulary of about 37000 tokens."

In [ ]:
tokenizer = train_tokenizer(config.data, max_samples=5000)
print(f"Vocabulary size: {tokenizer.vocab_size}")

loaders = get_dataloaders(
    config.data,
    tokenizer,
    splits=("train", "validation"),
    max_train_samples=2000,
    max_val_samples=500,
)

train_loader = loaders["train"]
val_loader = loaders["validation"]

print(f"Training batches: ~{len(train_loader)}")
print(f"Validation batches: ~{len(val_loader)}")

In [ ]:
src, tgt, src_mask, tgt_mask = next(iter(train_loader))
print(f"Batch shapes: src={src.shape}, tgt={tgt.shape}")
print(f"Mask shapes:  src_mask={src_mask.shape}, tgt_mask={tgt_mask.shape}")

for i in range(min(3, src.size(0))):
    src_text = tokenizer.decode(src[i].tolist())
    tgt_text = tokenizer.decode(tgt[i].tolist())
    print(f"\nExample {i + 1}:")
    print(f"  EN: {src_text}")
    print(f"  DE: {tgt_text}")

## 4. Model Architecture

Build the Transformer following Figure 1 of the paper: encoder (N layers of self-attention + FFN) and decoder (N layers of masked self-attention + cross-attention + FFN) with shared embeddings (Section 3.4).

In [ ]:
model = Transformer(
    vocab_size=tokenizer.vocab_size,
    d_model=config.model.d_model,
    n_layers=config.model.n_layers,
    n_heads=config.model.n_heads,
    d_ff=config.model.d_ff,
    dropout=config.model.dropout,
    max_len=config.data.max_seq_len,
    share_embeddings=config.model.share_embeddings,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Transformer parameters: {n_params:,} ({n_params / 1e6:.2f}M)")
print(f"  (Paper base model: ~65M parameters)")
print()

# Verify forward pass shapes
model.to(device)
test_src = src[:2].to(device)
test_tgt = tgt[:2, :-1].to(device)
test_src_mask = src_mask[:2].to(device)
test_tgt_mask = tgt_mask[:2].to(device)

with torch.no_grad():
    logits = model(test_src, test_tgt, test_src_mask, test_tgt_mask)

print(f"Forward pass verified:")
print(f"  Input:  src={test_src.shape}, tgt={test_tgt.shape}")
print(f"  Output: logits={logits.shape}")
print(f"  Expected: (batch, tgt_len, vocab_size={tokenizer.vocab_size})")

## 5. Positional Encoding Visualization

The sinusoidal positional encoding (Section 3.5) uses a geometric progression of wavelengths from $2\pi$ to $10000 \cdot 2\pi$:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [ ]:
pe = model.positional_encoding.pe[0, : config.data.max_seq_len, :].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap
im = axes[0].imshow(pe.T, aspect="auto", cmap="coolwarm", interpolation="nearest")
axes[0].set_xlabel("Position")
axes[0].set_ylabel("Dimension")
axes[0].set_title("Positional Encoding Matrix")
plt.colorbar(im, ax=axes[0])

# Individual dimension curves
for dim in [0, 1, 4, 5, config.model.d_model - 2, config.model.d_model - 1]:
    axes[1].plot(pe[:, dim], label=f"dim {dim}", alpha=0.8)
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Value")
axes[1].set_title("Positional Encoding by Dimension")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Training

Train using:
- **Adam optimizer**: $\beta_1=0.9$, $\beta_2=0.98$, $\epsilon=10^{-9}$ (Section 5.3)
- **Noam LR schedule**: linear warmup then inverse sqrt decay (Equation 5)
- **Label smoothing**: $\epsilon_{ls}=0.1$ (Section 5.4)
- **Gradient clipping**: max norm = 1.0

In [ ]:
history = train(
    model=model,
    train_dataloader=train_loader,
    vocab_size=tokenizer.vocab_size,
    d_model=config.model.d_model,
    warmup_steps=config.train.warmup_steps,
    total_steps=config.train.total_steps,
    padding_idx=PAD_IDX,
    label_smoothing=config.train.label_smoothing,
    log_interval=config.train.log_interval,
    device=device,
    checkpoint_dir=str(Path.cwd().parent / config.train.checkpoint_dir),
    checkpoint_interval=config.train.checkpoint_interval,
    val_dataloader=val_loader,
    val_interval=250,
    val_max_batches=20,
    tokenizer=tokenizer,
    bleu_samples=50,
    max_grad_norm=1.0,
)

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss
axes[0].plot(history.steps, history.losses, linewidth=1.5, label="Train")
if history.val_steps:
    axes[0].plot(history.val_steps, history.val_losses, "ro-", linewidth=2, label="Val")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate (Noam schedule)
axes[1].plot(history.steps, history.learning_rates, linewidth=1.5, color="orange")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Noam LR Schedule (Equation 5)")
axes[1].grid(True, alpha=0.3)

# BLEU during training
if history.val_bleu:
    axes[2].plot(history.val_steps, history.val_bleu, "gs-", linewidth=2)
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("BLEU")
    axes[2].set_title("Validation BLEU")
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].text(0.5, 0.5, "No BLEU data", ha="center", va="center", transform=axes[2].transAxes)
    axes[2].set_title("Validation BLEU")

plt.tight_layout()
plt.show()

## 8. Inference: Greedy and Beam Search Decoding

Translate sample sentences using greedy decoding and beam search (Section 6.1).

> "We used beam search with a beam size of 4 and length penalty alpha=0.6."

In [ ]:
model.eval()
val_batch = next(iter(val_loader))
src_val, tgt_val, src_mask_val, _ = val_batch
n_examples = min(5, src_val.size(0))

print("=" * 70)
print("Translation Examples")
print("=" * 70)

for i in range(n_examples):
    src_i = src_val[i : i + 1].to(device)
    src_mask_i = src_mask_val[i : i + 1].to(device)

    source = tokenizer.decode(src_val[i].tolist())
    ref = tokenizer.decode(tgt_val[i].tolist())

    # Greedy decoding
    greedy_ids = greedy_decode(
        model,
        src_i,
        src_mask_i,
        bos_token=BOS_IDX,
        eos_token=EOS_IDX,
        max_len=src_val.size(1) + 50,
    )
    greedy_text = tokenizer.decode(greedy_ids.tolist())

    # Beam search
    beam_ids = beam_search(
        model,
        src_i,
        src_mask_i,
        bos_token=BOS_IDX,
        eos_token=EOS_IDX,
        beam_size=config.inference.beam_size,
        max_len=src_val.size(1) + 50,
        length_penalty_alpha=config.inference.length_penalty_alpha,
    )
    beam_text = tokenizer.decode(beam_ids.tolist())

    print(f"\nExample {i + 1}:")
    print(f"  Source (EN):    {source}")
    print(f"  Reference (DE): {ref}")
    print(f"  Greedy:         {greedy_text}")
    print(f"  Beam (k={config.inference.beam_size}):    {beam_text}")

## 9. Attention Weight Visualization

Extract attention weight heatmaps using forward hooks on `ScaledDotProductAttention` modules. This lets us inspect what each head attends to without modifying the model code.

In [ ]:
# Register forward hooks to capture attention weights
attention_weights = {}


def make_hook(name):
    def hook_fn(module, input, output):
        # ScaledDotProductAttention returns (output, attn_weights)
        _, weights = output
        attention_weights[name] = weights.detach().cpu()

    return hook_fn


hooks = []
for name, module in model.named_modules():
    if isinstance(module, ScaledDotProductAttention):
        hooks.append(module.register_forward_hook(make_hook(name)))

# Forward pass on a single validation example
model.eval()
src_i = src_val[0:1].to(device)
tgt_i = tgt_val[0:1, :-1].to(device)
src_mask_i = src_mask_val[0:1].to(device)
_, tgt_mask_i = create_masks(src_val[0:1], tgt_val[0:1, :-1])
tgt_mask_i = tgt_mask_i.to(device)

with torch.no_grad():
    _ = model(src_i, tgt_i, src_mask_i, tgt_mask_i)

# Remove hooks
for h in hooks:
    h.remove()

print(f"Captured attention weights from {len(attention_weights)} layers:")
for name, w in attention_weights.items():
    print(f"  {name}: {w.shape}")

In [ ]:
# Categorize attention layers
enc_self_attn = {k: v for k, v in attention_weights.items() if "encoder" in k}
dec_self_attn = {
    k: v
    for k, v in attention_weights.items()
    if "decoder" in k and "self_attn" in k
}
dec_cross_attn = {
    k: v
    for k, v in attention_weights.items()
    if "decoder" in k and "cross_attn" in k
}


def plot_attention_heads(attn_dict, title_prefix, xlabel="Key Position", ylabel="Query Position"):
    """Plot attention heads for the first layer in the dict."""
    if not attn_dict:
        print(f"No {title_prefix} weights captured.")
        return
    key = list(attn_dict.keys())[0]
    weights = attn_dict[key][0]  # (n_heads, q_len, k_len)
    n_heads = weights.shape[0]

    fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
    if n_heads == 1:
        axes = [axes]
    for h in range(n_heads):
        im = axes[h].imshow(weights[h].numpy(), cmap="viridis", aspect="auto")
        axes[h].set_title(f"Head {h + 1}")
        axes[h].set_xlabel(xlabel)
        if h == 0:
            axes[h].set_ylabel(ylabel)
    fig.suptitle(f"{title_prefix} ({key})", fontsize=12)
    plt.tight_layout()
    plt.show()


plot_attention_heads(enc_self_attn, "Encoder Self-Attention")
plot_attention_heads(dec_self_attn, "Decoder Self-Attention (Masked)")
plot_attention_heads(
    dec_cross_attn,
    "Decoder Cross-Attention",
    xlabel="Source Position",
    ylabel="Target Position",
)

## 10. Validation BLEU Score

In [ ]:
bleu_score = compute_bleu(
    model=model,
    val_dataloader=val_loader,
    tokenizer=tokenizer,
    device=device,
    max_samples=100,
)

print(f"Validation BLEU: {bleu_score:.2f}")
print(f"  (Paper reports 27.3 BLEU for base model on WMT14 EN-DE)")
print(f"  (This is a quick_test config with tiny model and limited data)")

## 11. Key Observations

### Paper vs. Implementation

| Aspect | Paper (Base) | This Demo (quick_test) |
|--------|-------------|----------------------|
| d_model | 512 | 64 |
| Layers | 6 | 2 |
| Heads | 8 | 4 |
| Vocab size | ~37K | 1K |
| Training steps | 100K | 500 |
| Training data | 4.5M pairs | 2K pairs |
| BLEU (EN-DE) | 27.3 | ~1-5 |

The quick_test model is far too small and undertrained to produce meaningful translations. The purpose is to verify the pipeline end-to-end.

### Architecture Highlights Verified

1. **Shared embeddings** (Section 3.4): Input, output, and pre-softmax weights are tied
2. **Sinusoidal positional encoding** (Section 3.5): Geometric progression of wavelengths
3. **Noam LR schedule** (Equation 5): Linear warmup then inverse sqrt decay
4. **Label smoothing** (Section 5.4): $\epsilon_{ls}=0.1$ with KL divergence loss
5. **Token-count batching** (Section 5.1): Batches target ~25K tokens, sorted by length
6. **Beam search** (Section 6.1): beam_size=4, length penalty $\alpha=0.6$

### To reproduce full paper results:

```python
config = ExperimentConfig.default()  # Base model (d_model=512, N=6, h=8)
# Train tokenizer on full data (max_samples=1_000_000)
# Use full WMT14 dataset (~4.5M pairs, no max_train_samples limit)
# Train for 100K steps on 8 GPUs
# Average last 5 checkpoints for evaluation
```

## 12. Save Results

In [ ]:
import json

results_dir = Path.cwd().parent / "results"
results_dir.mkdir(exist_ok=True)

results = {
    "config": {
        "model": vars(config.model),
        "data": vars(config.data),
        "train": vars(config.train),
    },
    "training_steps": history.steps,
    "training_losses": history.losses,
    "learning_rates": history.learning_rates,
    "val_steps": history.val_steps,
    "val_losses": history.val_losses,
    "val_bleu": history.val_bleu,
    "final_bleu": bleu_score,
}

with open(results_dir / "experiment_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_dir / 'experiment_results.json'}")